In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

# --- Configuration ---
DATA_FILE = '../data/scenario_6_marina_features_50ms_final.csv'

# Parameters for slope-based labeling
LOOKAHEAD_WINDOW_MS = 500      # Time window to look into the future (0.5 seconds)
DEPLETING_THRESHOLD_MS = -100  # Buffer must drop by at least 100ms to be 'Depleting'
FILLING_THRESHOLD_MS = 100     # Buffer must rise by at least 100ms to be 'Filling'

# --- Load Data ---
df = pd.read_csv(DATA_FILE)
print("Dataset loaded successfully.")
df.head()

In [ ]:
print(f"Generating labels based on buffer slope over a {LOOKAHEAD_WINDOW_MS}ms window...")

# Ensure data is sorted chronologically to calculate trends
df.sort_values('timestamp', inplace=True)

# Calculate how many rows correspond to the lookahead window (50ms per row)
lookahead_steps = int(LOOKAHEAD_WINDOW_MS / 50)

# Get the future buffer level and calculate the change (slope)
df['future_buffer_level'] = df['buffer_level_ms'].shift(-lookahead_steps)
df['buffer_change'] = df['future_buffer_level'] - df['buffer_level_ms']

# Drop rows at the end where the future value is unknown
df.dropna(subset=['future_buffer_level'], inplace=True)

# Define conditions and assign labels
conditions = [
    df['buffer_change'] < DEPLETING_THRESHOLD_MS,
    df['buffer_change'] > FILLING_THRESHOLD_MS
]
choices = ['Depleting', 'Filling']
df['buffer_state'] = np.select(conditions, choices, default='Steady')

print("\nNew Label Distribution:")
print(df['buffer_state'].value_counts(normalize=True))

In [ ]:
# Prepare data for machine learning
all_features = [
    'packet_count', 'ps_sum', 'ps2_sum', 'ps3_sum', 
    'iat_sum', 'iat2_sum', 'iat3_sum'
]
X = df[all_features]
y = df['buffer_state']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Train the baseline model
model_baseline = RandomForestClassifier(
    max_depth=5, 
    random_state=42, 
    class_weight='balanced'
)
model_baseline.fit(X_train, y_train)

# Evaluate the model
print("--- Baseline Model (All Marina Features) Evaluation ---")
y_pred_baseline = model_baseline.predict(X_test)
print(classification_report(y_test, y_pred_baseline))

# Analyze and plot feature importances
importances = pd.Series(model_baseline.feature_importances_, index=all_features)
importances.sort_values(ascending=False, inplace=True)

print("\n--- Feature Importances from Baseline Model ---")
print(importances)

plt.figure(figsize=(10, 6))
importances.sort_values().plot(kind='barh', title='Feature Importances for Predicting Buffer Trend')
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
feature_experiments = {
    "First_Moment_Only": ['packet_count', 'ps_sum', 'iat_sum'],
    "Packet_Size_Features_Only": ['packet_count', 'ps_sum', 'ps2_sum', 'ps3_sum'],
    "IAT_Features_Only": ['packet_count', 'iat_sum', 'iat2_sum', 'iat3_sum'],
    "All_Marina_Features": all_features
}

results = {}

print("\n--- Running Feature Set Experiments ---")

for name, features_to_test in feature_experiments.items():
    print(f"\n--- Testing: {name} ---")
    
    # Prepare data with the specific feature subset
    X_exp = df[features_to_test]
    X_train_exp, X_test_exp, y_train_exp, y_test_exp = train_test_split(
        X_exp, y, test_size=0.25, random_state=42, stratify=y
    )
    
    # Train and evaluate a new model for the experiment
    model = RandomForestClassifier(max_depth=5, random_state=42, class_weight='balanced')
    model.fit(X_train_exp, y_train_exp)
    y_pred_exp = model.predict(X_test_exp)
    report = classification_report(y_test_exp, y_pred_exp, output_dict=True, zero_division=0)
    
    print(classification_report(y_test_exp, y_pred_exp, zero_division=0))
    
    if 'Depleting' in report:
        results[name] = report['Depleting']['recall']

print("\n\n--- Final Results: Recall for 'Depleting' State Across Experiments ---")
results_series = pd.Series(results)
results_series.sort_values(ascending=False, inplace=True)
print(results_series)